In [ ]:
" Part 2: Price Wars "
" Execution time: ~ 1.5 minutes "

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Parameters
T = 1000
K = 5
beta = 1
epochs = 20
epsilon = (K * np.log(K) / T)**(1/3)
eta = epsilon
window = 100

# Create discrete set of K price levels
base_seed = 63
np.random.seed(base_seed)
prices_set = np.random.uniform(0, 5, K)
max_price = np.max(prices_set)
min_price = np.min(prices_set)

# Find myopic best response to UCB
def find_optimal_q_UCB(p_t, beta):
    # Objective function to MINIMIZE
    def objective(q_array):
        q = q_array[0]
        prob_choose_competitor = 1.0 - (1.0 / (1.0 + np.exp(beta * (p_t - q))))
        expected_reward = q * prob_choose_competitor
        return -expected_reward

    # Run optimization with bounds [0, infinity)
    result = minimize(objective, x0=[p_t], bounds=[(0, None)])

    return result.x[0] # Returns the exact continuous price q_t

# Find myopic best response to MW
def find_optimal_q_MW(q_k, prices_set, beta):
    def objective(q_array):
        q_t = q_array[0]
        current_expected_reward = 0

        for i in range(len(prices_set)):
            p_k = prices_set[i]
            prob_p_k = q_k[i]

            prob_choose_competitor = 1.0 - (1.0 / (1.0 + np.exp(beta * (p_k - q_t))))
            current_expected_reward += prob_p_k * (q_t * prob_choose_competitor)

        return -current_expected_reward

    # Start search near the weighted average of the learner's possible prices
    expected_p = np.sum(q_k * prices_set)

    # Run optimization with bounds [0, infinity)
    result = minimize(objective, x0=[expected_p], bounds=[(0, None)])

    return result.x[0] # Returns the exact continuous price q_t

# Arrays to store results over epochs for MC Mean
all_cum_learner_UCB = np.zeros((epochs, T))
all_cum_competitor_UCB = np.zeros((epochs, T))
all_cum_learner_MW = np.zeros((epochs, T))
all_cum_competitor_MW = np.zeros((epochs, T))

all_prices_learner_UCB = np.zeros((epochs, T))
all_prices_competitor_UCB = np.zeros((epochs, T))
all_prices_learner_MW = np.zeros((epochs, T))
all_prices_competitor_MW = np.zeros((epochs, T))

# Customer Choices (1 = Learner Won, 0 = Competitor Won)
all_choices_UCB = np.zeros((epochs, T))
all_choices_MW = np.zeros((epochs, T))

# Arm Selections (Track how many times each arm is pulled per epoch)
all_pulls_UCB = np.zeros((epochs, K))
all_pulls_MW = np.zeros((epochs, K))

# MONTE CARLO
for ep in range(epochs):

    np.random.seed(base_seed + ep)

    # ======================
    # 1. UCB
    # ======================

    # Arrays to track rewards
    learner_rewards_UCB = np.zeros(T)
    competitor_rewards_UCB = np.zeros(T)

    Npull = np.zeros(K)
    sum_rewards = np.zeros(K)

    for t in range(T):
        # Select arm
        if t < K:
          arm = t
        else:
          exploit = sum_rewards / Npull
          explore = (max_price - min_price)*np.sqrt((2*np.log(T)) / Npull)
          ucb = exploit + explore
          arm = np.argmax(ucb)

        all_pulls_UCB[ep, arm] += 1

        # Get p_t
        p_t = prices_set[arm]

        # Find myopic best response
        q_t = find_optimal_q_UCB(p_t,beta)

        # Store prices for this round
        all_prices_learner_UCB[ep, t] = p_t
        all_prices_competitor_UCB[ep, t] = q_t

        # Customer probabilistic choice
        prob_choose_learner = 1 / (1 + np.exp(beta * (p_t - q_t)))
        if np.random.random() < prob_choose_learner:
            reward_l, reward_c = p_t, 0   # Customer chose learner
            all_choices_UCB[ep, t] = 1
        else:
            reward_l, reward_c = 0, q_t   # Customer chose competitor
            all_choices_UCB[ep, t] = 0

        # Track rewards
        learner_rewards_UCB[t] = reward_l
        competitor_rewards_UCB[t] = reward_c

        # Update UCB terms
        Npull[arm] += 1
        sum_rewards[arm] += reward_l

    # Store the cumulative sum for this epoch
    all_cum_learner_UCB[ep] = np.cumsum(learner_rewards_UCB)
    all_cum_competitor_UCB[ep] = np.cumsum(competitor_rewards_UCB)

    # ======================
    # 2. MW
    # ======================

    # Arrays to track rewards
    learner_rewards_MW = np.zeros(T)
    competitor_rewards_MW = np.zeros(T)
    w = np.ones(K)

    for t in range(T):
        # Probability distribution from weights
        p = w / np.sum(w)

        # Adding exploration
        q = (1 - epsilon)*p + epsilon/K

        # Select arm randomly according to MW probabilities
        arm = np.random.choice(K, p=q)
        all_pulls_MW[ep, arm] += 1

        # Get p_t
        p_t = prices_set[arm]

        # Find myopic best response
        q_t = find_optimal_q_MW(q,prices_set,beta)

        # Store prices for this round
        all_prices_learner_MW[ep, t] = p_t
        all_prices_competitor_MW[ep, t] = q_t

        # Customer probabilistic choice
        prob_choose_learner = 1 / (1 + np.exp(beta * (p_t - q_t)))
        if np.random.random() < prob_choose_learner:
            reward_l, reward_c = p_t, 0   # Customer chose learner
            all_choices_MW[ep, t] = 1
        else:
            reward_l, reward_c = 0, q_t   # Customer chose competitor
            all_choices_MW[ep, t] = 0

        # Track rewards
        learner_rewards_MW[t] = reward_l
        competitor_rewards_MW[t] = reward_c

        # Estimated loss vector
        loss_hat = np.zeros(K)
        loss_hat[arm] = (1 - (reward_l / max_price)) / q[arm]

        # MW update
        w = w * ((1 - eta) ** loss_hat)

    # Store the cumulative sum for this epoch
    all_cum_learner_MW[ep] = np.cumsum(learner_rewards_MW)
    all_cum_competitor_MW[ep] = np.cumsum(competitor_rewards_MW)

# Calculate mean pulls across all epochs
mean_pulls_UCB = np.mean(all_pulls_UCB, axis=0)
mean_pulls_MW = np.mean(all_pulls_MW, axis=0)

# Sort prices for cleaner display
sorted_indices = np.argsort(prices_set)
sorted_prices = prices_set[sorted_indices]
sorted_pulls_UCB = mean_pulls_UCB[sorted_indices]
sorted_pulls_MW = mean_pulls_MW[sorted_indices]

# Rewards MC Mean
mean_learner_UCB = np.mean(all_cum_learner_UCB, axis=0)
mean_competitor_UCB = np.mean(all_cum_competitor_UCB, axis=0)
mean_learner_MW = np.mean(all_cum_learner_MW, axis=0)
mean_competitor_MW = np.mean(all_cum_competitor_MW, axis=0)

# Final Output Stats
# UCB
final_l_mean_UCB = mean_learner_UCB[-1]
final_c_mean_UCB = mean_competitor_UCB[-1]
final_l_std_UCB = np.std(all_cum_learner_UCB[:, -1])
final_c_std_UCB = np.std(all_cum_competitor_UCB[:, -1])
# MW
final_l_mean_MW = mean_learner_MW[-1]
final_c_mean_MW = mean_competitor_MW[-1]
final_l_std_MW = np.std(all_cum_learner_MW[:, -1])
final_c_std_MW = np.std(all_cum_competitor_MW[:, -1])

# Prices MC Mean
mean_price_l_UCB = np.mean(all_prices_learner_UCB, axis=0)
mean_price_c_UCB = np.mean(all_prices_competitor_UCB, axis=0)
mean_price_l_MW = np.mean(all_prices_learner_MW, axis=0)
mean_price_c_MW = np.mean(all_prices_competitor_MW, axis=0)

# Choices MC Mean (Win Rates)
mean_choices_UCB = np.mean(all_choices_UCB, axis=0)
mean_choices_MW = np.mean(all_choices_MW, axis=0)

# Function to optionally smooth prices for cleaner plots
def smooth(y, box_pts):
    box = np.ones(box_pts)/box_pts
    return np.convolve(y, box, mode='valid')

# Calculate smoothed frequencies
smooth_learner_freq_UCB = smooth(mean_choices_UCB, window)
smooth_comp_freq_UCB = 1.0 - smooth_learner_freq_UCB

smooth_learner_freq_MW = smooth(mean_choices_MW, window)
smooth_comp_freq_MW = 1.0 - smooth_learner_freq_MW

print("=" * 60)
print(f"AVERAGE ARM SELECTIONS (MC Mean over {epochs} epochs)")
print("=" * 60)
print(f"{'Price Level':<15} | {'UCB Pulls':<15} | {'MW Pulls':<15}")
print("-" * 60)
for i in range(K):
    print(f"${sorted_prices[i]:<14.2f} | {sorted_pulls_UCB[i]:<15.1f} | {sorted_pulls_MW[i]:<15.1f}")
print("=" * 60)

print("-" * 54)
print(f"[UCB] Learner MC Mean Final Reward:    {final_l_mean_UCB:.2f} ± {final_l_std_UCB:.2f}")
print(f"[UCB] Competitor MC Mean Final Reward: {final_c_mean_UCB:.2f} ± {final_c_std_UCB:.2f}")
print("-" * 54)
print(f"[MW] Learner MC Mean Final Reward:    {final_l_mean_MW:.2f} ± {final_l_std_MW:.2f}")
print(f"[MW] Competitor MC Mean Final Reward: {final_c_mean_MW:.2f} ± {final_c_std_MW:.2f}")
print("-" * 54)

# PLOTTING
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 12))
# Subplot 1: Cumulative Reward Curves
ax1.plot(mean_learner_UCB, label='Learner (UCB)', color='blue', linewidth=1.5)
ax1.plot(mean_competitor_UCB, label='Competitor (vs UCB)', color='red', linestyle='--', linewidth=1.5)
ax1.plot(mean_learner_MW, label='Learner (MW)', color='green', linewidth=1.5)
ax1.plot(mean_competitor_MW, label='Competitor (vs MW)', color='orange', linestyle='--', linewidth=1.5)

ax1.set_title(f'Cumulative Reward Curves (MC Mean) [$\\beta$ = {beta}]')
ax1.set_xlabel('Time Step (t)')
ax1.set_ylabel('Cumulative Reward')
ax1.set_xlim(0, T)
ax1.grid(True, linestyle=":", alpha=0.6)

# Subplot 2: Prices Over Time (Smoothed)
t_plot = np.arange(window - 1, T)
ax2.plot(t_plot, smooth(mean_price_l_UCB, window), color='blue', alpha=0.8)
ax2.plot(t_plot, smooth(mean_price_c_UCB, window), color='red', linestyle='--', alpha=0.8)
ax2.plot(t_plot, smooth(mean_price_l_MW, window), color='green', alpha=0.8)
ax2.plot(t_plot, smooth(mean_price_c_MW, window), color='orange', linestyle='--', alpha=0.8)

ax2.set_title(f'Prices Over Time (MC Mean, Smoothed) [$\\beta$ = {beta}]')
ax2.set_xlabel('Time Step (t)')
ax2.set_ylabel('Price')
ax2.set_xlim(window - 1, T)
ax2.grid(True, linestyle=":", alpha=0.6)

# Subplot 3: UCB Frequencies
ax3.plot(t_plot, smooth_learner_freq_UCB, color='blue', linewidth=1.5)
ax3.plot(t_plot, smooth_comp_freq_UCB, color='red', linestyle='--', linewidth=1.5)
ax3.set_title(f'Rolling Choice Frequency (UCB vs Myopic) [$\\beta$ = {beta}]')
ax3.set_xlabel('Time Step (t)')
ax3.set_ylabel('Win Frequency (0 to 1)')
ax3.set_ylim(0, 1.05)
ax3.set_xlim(window - 1, T)
ax3.grid(True, linestyle=":", alpha=0.6)

# Subplot 4: MW Frequencies
ax4.plot(t_plot, smooth_learner_freq_MW, color='green', linewidth=1.5)
ax4.plot(t_plot, smooth_comp_freq_MW, color='orange', linestyle='--', linewidth=1.5)
ax4.set_title(f'Rolling Choice Frequency (MW vs Myopic) [$\\beta$ = {beta}]')
ax4.set_xlabel('Time Step (t)')
ax4.set_ylabel('Win Frequency (0 to 1)')
ax4.set_ylim(0, 1.05)
ax4.set_xlim(window - 1, T)
ax4.grid(True, linestyle=":", alpha=0.6)

# Master Legend & Layout
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, bbox_to_anchor=(0.5, 0.01))
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()